# D1 EDA — what the two datasets actually contain

**DTSC3602 Group 9** · Evan Smith, Andrew Wallace, Ikumi Uemura, Emmett Lashbrook, Brian Aguirre

Before any modelling, three questions:

1. How complete are the fields the recommender wants to use?
2. How do the 2,483 resumes distribute across the 24 categories?
3. How much of the posting corpus can our taxonomy actually label — the weak-supervision coverage rate, which caps how much training data exists?

Run `python -m src.data_load --rebuild` first so the parquet caches exist.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import viz
from src.data_load import load_postings, load_resumes
from src.pairs import assign_posting_categories

viz.use_style()

postings = assign_posting_categories(load_postings())
resumes = load_resumes()

print(f"postings  {len(postings):,}")
print(f"resumes   {len(resumes):,}  across {resumes.category.nunique()} categories")

## 1. Field completeness

This drives a concrete modelling decision. Salary is absent on ~71% of postings and experience level on ~24%, so the feature builder carries explicit `*_missing` indicators instead of imputing. Imputing a median salary would invent agreement with a user's target that the data never recorded.

In [ ]:
fields = {
    "title": postings.title.notna(),
    "description": postings.posting_text.str.len() > 50,
    "work type": postings.formatted_work_type.notna(),
    "skills tagged": postings.skill_abrs.map(len) > 0,
    "industries tagged": postings.industries.map(len) > 0,
    "company": postings.company_official_name.notna(),
    "state parsed": postings.state.notna(),
    "experience level": postings.experience_rank.notna(),
    "salary": postings.salary_annual.notna(),
    "remote flagged": postings.remote_allowed.astype(bool),
}
coverage = pd.Series({k: 100 * v.mean() for k, v in fields.items()}).sort_values()

fig, ax = plt.subplots(figsize=(7, 4.2))
bars = ax.barh(coverage.index, coverage.values, color=viz.CATEGORICAL[0], height=0.68)
viz.bar_labels(ax, bars, fmt="%.0f%%")
ax.set_xlim(0, 112)
ax.set_xlabel("% of postings present")
ax.set_title("Field completeness across 123,781 postings")
ax.xaxis.grid(True); ax.yaxis.grid(False)
viz.despine(ax)
plt.tight_layout(); plt.show()

## 2. Resume categories

The resume set is close to balanced by design — most categories hold 95–120 records. The thin tail (BPO, AUTOMOBILE, AGRICULTURE) is worth remembering when reading the fairness table later: those categories have the least evidence behind them.

In [ ]:
counts = resumes.category.value_counts().sort_values()

fig, ax = plt.subplots(figsize=(7, 6))
bars = ax.barh(counts.index, counts.values, color=viz.CATEGORICAL[0], height=0.72)
viz.bar_labels(ax, bars)
ax.set_xlim(0, counts.max() * 1.13)
ax.set_xlabel("resumes")
ax.set_title("Resumes per category")
ax.xaxis.grid(True); ax.yaxis.grid(False)
viz.despine(ax)
plt.tight_layout(); plt.show()

## 3. Weak-label coverage — the number that caps our training data

`src/taxonomy.py` scores each posting against all 24 categories using title patterns (weight 3), skill abbreviations (2) and industry names (1), and assigns the best one if it clears a threshold of 2.0. Postings below the threshold get no label and are excluded from training.

Two things to read here: the **coverage rate**, and how **unevenly** the labelled postings spread across categories. The posting corpus is not balanced the way the resume set is — it reflects what was actually hiring on LinkedIn in 2023–24.

In [ ]:
assigned = postings.posting_category.notna()
print(f"labelled: {assigned.sum():,} / {len(postings):,}  ({assigned.mean():.1%})")

cat_counts = postings.loc[assigned, "posting_category"].value_counts().sort_values()

fig, ax = plt.subplots(figsize=(7, 6))
bars = ax.barh(cat_counts.index, cat_counts.values, color=viz.CATEGORICAL[0], height=0.72)
viz.bar_labels(ax, bars, fmt="{:,.0f}")
ax.set_xlim(0, cat_counts.max() * 1.18)
ax.set_xlabel("postings assigned")
ax.set_title("Postings per weak-labelled category")
ax.xaxis.grid(True); ax.yaxis.grid(False)
viz.despine(ax)
plt.tight_layout(); plt.show()

ratio = cat_counts.max() / cat_counts.min()
print(f"largest category is {ratio:.0f}x the smallest — pair sampling draws a "
      f"fixed quota per resume so this imbalance does not propagate into training.")

### What the unlabelled 40% looks like

Worth confirming these are genuinely ambiguous postings rather than a systematic gap in the taxonomy.

In [ ]:
unlabelled = postings.loc[~assigned]
print(f"{len(unlabelled):,} postings carry no category. Most common titles:\n")
print(unlabelled.title.str.lower().str.strip().value_counts().head(15).to_string())

print("\nscore distribution among unlabelled (threshold = 2.0):")
print(unlabelled.category_score.describe().round(2).to_string())

## 4. Salary and experience

Only the ~29% of postings with a salary contribute to the salary feature. Plotting that subset alone would overstate how much of the corpus the feature actually reaches, so the count is stated alongside.

In [ ]:
salary = postings.salary_annual.dropna()
salary = salary[(salary > 10_000) & (salary < 400_000)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.hist(salary, bins=60, color=viz.CATEGORICAL[0])
ax1.set_xlabel("annualised salary (USD)")
ax1.set_ylabel("postings")
ax1.set_title(f"Salary, where listed (n={len(salary):,})")
ax1.xaxis.set_major_formatter(lambda x, _: f"${x/1000:.0f}k")
ax1.xaxis.grid(False)
viz.despine(ax1)

order = ["Internship", "Entry level", "Associate", "Mid-Senior level", "Director", "Executive"]
levels = postings.formatted_experience_level.value_counts().reindex(order).fillna(0)
bars = ax2.bar(range(len(levels)), levels.values, color=viz.CATEGORICAL[0], width=0.7)
viz.bar_labels(ax2, bars, fmt="{:,.0f}")
ax2.set_xticks(range(len(levels)))
ax2.set_xticklabels(order, rotation=30, ha="right")
ax2.set_ylim(0, levels.max() * 1.15)
ax2.set_ylabel("postings")
ax2.set_title(f"Experience level, where stated (n={int(levels.sum()):,})")
ax2.xaxis.grid(False)
viz.despine(ax2)

plt.tight_layout(); plt.show()

print(f"median listed salary: ${salary.median():,.0f}")
print(f"postings with no experience level: {postings.experience_rank.isna().sum():,}")

## 5. Document lengths

Both corpora are truncated to 2,000 characters before embedding, and the encoder's own window caps each document at 256 word-pieces. Because `posting_text` is built as *title + company + skills + description*, the most identifying content sits at the front and survives truncation.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.hist(resumes.resume_text_full.str.len().clip(upper=20_000), bins=60,
        color=viz.CATEGORICAL[0], alpha=0.85, label="resumes (full text)")
ax.axvline(2000, color=viz.CATEGORICAL[1], linestyle="--", linewidth=2,
           label="2,000-char truncation")
ax.set_xlabel("characters"); ax.set_ylabel("documents")
ax.set_title("Resume length vs the truncation point")
ax.legend()
ax.xaxis.grid(False)
viz.despine(ax)
plt.tight_layout(); plt.show()

full = resumes.resume_text_full.str.len()
print(f"median resume: {full.median():,.0f} chars; {(full > 2000).mean():.0%} exceed the cutoff")

## Takeaways for the write-up

1. **Text fields are near-complete; structured fields are not.** Title, description, work type and the skill/industry tags are present on ~99% of postings, but salary reaches only ~29% and the remote flag ~12%. The recommender therefore leans on text, and treats preference fields as bonuses with explicit missingness indicators.
2. **Weak labelling covers ~59% of the corpus.** That is the ceiling on labelled training data. The unlabelled remainder is dominated by generic titles that genuinely do not map to one of 24 resume categories.
3. **The posting corpus is heavily skewed** toward healthcare, sales, IT and engineering, while the resume set is near-balanced. Pair sampling draws a fixed quota per resume so this skew does not leak into the class balance.
4. **Most resumes exceed the truncation point**, so the embedding sees roughly the first half of a typical resume. Front-loading the most identifying content is what makes that acceptable.